# Pairwise re-query — pilot (format A/B) → auto-select → 4-vote main run

One GPU session, one model load (base Qwen3-VL-8B + `qwen3vl_final` LoRA).
For each of the 2,421 prediction-adjacent high-cosine trigger pairs from the
1,914-sample holdout, asks the model pairwise which frame comes first per the
caption:

1. **Pilot** — 200 fixed-seed trigger pairs, formats A (natural FIRST/SECOND
   question) and B (training-format 2-image list), greedy, 1 query each.
   The winning format (tie within ±2%p → A) is chosen automatically; if both
   formats break down (>20% parse failures) the run aborts for review.
2. **Main** — all 2,421 pairs × 4 votes (2 per presentation order,
   temperature 0.7, per-batch sha256 seeds recorded in every row).

Inputs on Drive: `snu ai/snu_colab_full_train_bundle.tar` (existing),
`snu ai/snu_colab_pairwise_requery_bundle.tar` (code bundle, uploaded fresh),
`snu ai/snu_qwen3vl_outputs/qwen3vl_final/` (LoRA adapter, existing).

Outputs on Drive (`snu ai/outputs/pairwise_requery/`, resumable via
`votes_state.jsonl`): `votes.csv`, `metadata.json`, `pilot_votes.csv`,
`pilot_metadata.json`. The tau × rule sweep runs locally afterwards
(`scripts/sweep_pairwise_requery.py`) — no scoring happens here.


In [1]:
import os
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['HF_HUB_DISABLE_XET'] = '1'

DRIVE_ROOT = Path('/content/drive/MyDrive/snu ai')
CODE_BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_pairwise_requery_bundle.tar'
EXPECTED_BUNDLE_SHA256 = '2C261AA7F8B74797933DC7547B3E6D9A9B227696CD57D8D3DEA3328C31297D70'
WORKDIR = Path('/content/snu_ai')
DATA_ROOT = WORKDIR / 'snuaichallenge_data'
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGE_DIR = DATA_ROOT / 'train'
TRIGGER_CSV = WORKDIR / 'outputs/pairwise_requery/trigger_pairs.csv'
TRIGGER_METADATA = WORKDIR / 'outputs/pairwise_requery/trigger_metadata.json'
HOLDOUT_INPUT = WORKDIR / 'outputs/hybrid_judgment/holdout_input.csv'

BASE_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
ADAPTER_DIR = DRIVE_ROOT / 'snu_qwen3vl_outputs/qwen3vl_final'
OUT_DIR = DRIVE_ROOT / 'outputs/pairwise_requery'

HF_HOME = Path('/content/huggingface')
os.environ['HF_HOME'] = str(HF_HOME)

N_PAIRS_EXPECTED = 2421
N_VOTES = 4
PILOT_N = 200
PILOT_SEED = 20260721
SEED_BASE = 20260721
TEMPERATURE = 0.7
PAIR_BATCH_SIZE = 16
TIE_MARGIN = 0.02
PARSE_FAIL_ABORT = 0.20


## 1. Mount Drive, install dependencies, extract the two bundles

In [2]:
import hashlib
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0',
    'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0',
], check=True)


def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None


WORKDIR.mkdir(parents=True, exist_ok=True)
if not TRAIN_CSV.is_file() or not TRAIN_IMAGE_DIR.is_dir():
    train_bundle = find_drive_file('snu_colab_full_train_bundle.tar')
    if train_bundle is None:
        raise FileNotFoundError('snu_colab_full_train_bundle.tar was not found in MyDrive')
    print(f'Extracting train bundle: {train_bundle}')
    subprocess.run(['tar', '-xf', str(train_bundle), '-C', str(WORKDIR)], check=True)

if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Upload the code bundle to this exact path: {CODE_BUNDLE_PATH}')
bundle_sha256 = hashlib.sha256(CODE_BUNDLE_PATH.read_bytes()).hexdigest().upper()
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(f'Bundle checksum mismatch: {bundle_sha256} != {EXPECTED_BUNDLE_SHA256}')
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)

import json

for required in (TRIGGER_CSV, TRIGGER_METADATA, HOLDOUT_INPUT):
    if not required.is_file():
        raise FileNotFoundError(required)
if not (ADAPTER_DIR / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'LoRA adapter not found on Drive: {ADAPTER_DIR}')
if not (ADAPTER_DIR / 'adapter_model.safetensors').is_file():
    raise FileNotFoundError(f'adapter_model.safetensors missing in {ADAPTER_DIR}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKDIR))

trigger_metadata = json.loads(TRIGGER_METADATA.read_text(encoding='utf-8'))
assert trigger_metadata['sanity_gate_passed'], trigger_metadata['sanity']
assert trigger_metadata['n_pairs'] == N_PAIRS_EXPECTED, trigger_metadata['n_pairs']
print('Sanity figures (re-derived locally before bundling):', trigger_metadata['sanity'])
print('Setup complete. Code bundle sha256:', bundle_sha256)


Mounted at /content/drive
Extracting train bundle: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
Sanity figures (re-derived locally before bundling): {'n_holdout': 1914, 'n_errors': 900, 'n_adjacent_transposition': 349, 'swapped_pair_cos_mean': 0.8653}
Setup complete. Code bundle sha256: 2C261AA7F8B74797933DC7547B3E6D9A9B227696CD57D8D3DEA3328C31297D70


## 2. Load the model once (base + LoRA)

Same pinned load as the holdout inference run; both the pilot and the main
run reuse this single instance.

In [3]:
import time

import torch

gpu_name = torch.cuda.get_device_name(0)
total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({total_vram_gib:.1f} GiB)')

from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, revision=MODEL_REVISION)
processor.tokenizer.padding_side = 'left'
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
load_seconds = time.perf_counter() - load_started
lora_b_mean = None
for name, param in model.named_parameters():
    if 'lora_B' in name:
        lora_b_mean = param.abs().mean().item()
        break
print(f'LoRA model loaded in {load_seconds:.1f}s; lora_B mean={lora_b_mean}')
assert lora_b_mean and lora_b_mean > 0, 'adapter weights look untrained or empty'


GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

LoRA model loaded in 135.0s; lora_B mean=0.008305066265165806


## 3. Pilot — format A vs B on 200 fixed-seed pairs (greedy)

Risk being measured: the LoRA is format-locked to 4-element list outputs, so
format A (unfamiliar question) may break down, while format B (its training
prompt shrunk to 2 images) may relapse into 4-element lists. Accuracy is
scored against the GT-earlier frame carried in trigger_pairs.csv; overall
accuracy (parse failure = wrong) decides, tie within TIE_MARGIN goes to A.

In [4]:
import random

from snu_ordering.pairwise_requery import (
    VOTE_FIELDNAMES,
    build_vote_requests,
    load_trigger_pairs,
    read_csv_rows,
    run_pair_inference,
    write_csv,
)

trigger_pairs = load_trigger_pairs(TRIGGER_CSV)
assert len(trigger_pairs) == N_PAIRS_EXPECTED, len(trigger_pairs)
inputs_by_id = {row['Id']: row for row in read_csv_rows(HOLDOUT_INPUT)}
missing_ids = sorted({p['Id'] for p in trigger_pairs} - set(inputs_by_id))
assert not missing_ids, missing_ids[:5]

gt_first = {
    (p['Id'], p['frame_a'], p['frame_b']): p['gt_first_frame']
    for p in trigger_pairs
}
pilot_pairs = random.Random(PILOT_SEED).sample(trigger_pairs, PILOT_N)
pilot_requests = build_vote_requests(pilot_pairs, inputs_by_id, n_votes=1)

pilot_stats = {}
pilot_rows = []
for prompt_format in ('A', 'B'):
    records = run_pair_inference(
        processor, model, pilot_requests, str(TRAIN_IMAGE_DIR),
        prompt_format=prompt_format, batch_size=PAIR_BATCH_SIZE,
        state_path=OUT_DIR / f'pilot_state_{prompt_format}.jsonl',
        do_sample=False, progress_every=100,
    )
    n_fail = sum(not r['parse_ok'] for r in records)
    n_correct = sum(
        r['parse_ok']
        and r['parsed_answer'] == gt_first[(r['Id'], r['frame_a'], r['frame_b'])]
        for r in records
    )
    pilot_stats[prompt_format] = {
        'n': len(records),
        'parse_fail_rate': round(n_fail / len(records), 4),
        'accuracy_overall': round(n_correct / len(records), 4),
        'accuracy_parsed_only': (
            round(n_correct / (len(records) - n_fail), 4)
            if n_fail < len(records) else None
        ),
    }
    for record in records:
        correct = int(
            record['parse_ok']
            and record['parsed_answer']
            == gt_first[(record['Id'], record['frame_a'], record['frame_b'])]
        )
        pilot_rows.append(dict(record, format=prompt_format, correct=correct))
    print(prompt_format, pilot_stats[prompt_format])

write_csv(
    OUT_DIR / 'pilot_votes.csv',
    ['format'] + VOTE_FIELDNAMES + ['correct'],
    pilot_rows,
)

if all(s['parse_fail_rate'] > PARSE_FAIL_ABORT for s in pilot_stats.values()):
    raise RuntimeError(
        f'Both formats exceed {PARSE_FAIL_ABORT:.0%} parse failures — '
        f'stopping for review: {pilot_stats}'
    )

acc_a = pilot_stats['A']['accuracy_overall']
acc_b = pilot_stats['B']['accuracy_overall']
CHOSEN_FORMAT = 'A' if acc_a >= acc_b - TIE_MARGIN else 'B'
(OUT_DIR / 'pilot_metadata.json').write_text(json.dumps({
    'pilot_n': PILOT_N,
    'pilot_seed': PILOT_SEED,
    'stats': pilot_stats,
    'chosen_format': CHOSEN_FORMAT,
    'tie_rule': f'A unless B leads by more than {TIE_MARGIN:.0%} overall accuracy',
}, indent=2) + '\n', encoding='utf-8')
print()
print(f'CHOSEN_FORMAT = {CHOSEN_FORMAT}  (A={acc_a:.4f} vs B={acc_b:.4f})')


A {'n': 200, 'parse_fail_rate': 0.0, 'accuracy_overall': 0.695, 'accuracy_parsed_only': 0.695}
B {'n': 200, 'parse_fail_rate': 0.14, 'accuracy_overall': 0.58, 'accuracy_parsed_only': 0.6744}

CHOSEN_FORMAT = A  (A=0.6950 vs B=0.5800)


## 4. Main run — 2,421 pairs × 4 votes, temperature 0.7

Votes 0–1 present (a, b), votes 2–3 present (b, a) to control position bias.
State appends to Drive (`votes_state.jsonl`), so a dead session resumes for
free; each row records the sha256-derived seed its batch was sampled with.

In [5]:
vote_requests = build_vote_requests(trigger_pairs, inputs_by_id, n_votes=N_VOTES)
assert len(vote_requests) == N_PAIRS_EXPECTED * N_VOTES, len(vote_requests)

main_started = time.perf_counter()
vote_records = run_pair_inference(
    processor, model, vote_requests, str(TRAIN_IMAGE_DIR),
    prompt_format=CHOSEN_FORMAT, batch_size=PAIR_BATCH_SIZE,
    state_path=OUT_DIR / 'votes_state.jsonl',
    do_sample=True, temperature=TEMPERATURE, seed_base=SEED_BASE,
    progress_every=500,
)
main_wall_seconds = time.perf_counter() - main_started

write_csv(OUT_DIR / 'votes.csv', VOTE_FIELDNAMES, vote_records)

import transformers

gen_times = sorted(record['gen_time_ms'] for record in vote_records)
timing = {
    'mean_ms': sum(gen_times) / len(gen_times),
    'median_ms': gen_times[len(gen_times) // 2],
    'p90_ms': gen_times[int(len(gen_times) * 0.9)],
    'max_ms': gen_times[-1],
}
(OUT_DIR / 'metadata.json').write_text(json.dumps({
    'gpu': gpu_name,
    'base_model': f'{BASE_MODEL_ID}@{MODEL_REVISION}',
    'adapter_dir': str(ADAPTER_DIR),
    'transformers_version': transformers.__version__,
    'chosen_format': CHOSEN_FORMAT,
    'pilot': pilot_stats,
    'n_pairs': N_PAIRS_EXPECTED,
    'n_votes': N_VOTES,
    'do_sample': True,
    'temperature': TEMPERATURE,
    'top_p': model.generation_config.top_p,
    'top_k': model.generation_config.top_k,
    'seed_base': SEED_BASE,
    'batch_size': PAIR_BATCH_SIZE,
    'seeding': 'sha256(seed_base|format|first_key|batch_len) per generate '
               'call, recorded per vote; a resume shifts batch boundaries',
    'request_order': 'trigger_pairs.csv row order x vote_idx 0-3; votes 0-1 '
                     'present (a,b), votes 2-3 present (b,a)',
    'parse_failures': sum(not r['parse_ok'] for r in vote_records),
    'model_load_seconds': round(load_seconds, 1),
    'run_wall_seconds': round(main_wall_seconds, 1),
    'gen_time_ms': {key: round(value, 1) for key, value in timing.items()},
}, indent=2) + '\n', encoding='utf-8')
print('Wrote', OUT_DIR / 'votes.csv')
print('Per-vote gen time:', {key: round(value, 1) for key, value in timing.items()})


Wrote /content/drive/MyDrive/snu ai/outputs/pairwise_requery/votes.csv
Per-vote gen time: {'mean_ms': 61.0, 'median_ms': 61.8, 'p90_ms': 67.4, 'max_ms': 98.2}


## 5. Validate before leaving the session

Row counts, parse-failure and contradiction rates per presentation order — a
large ab/ba contradiction gap means position bias survived the ordering
control and the sweep results need extra scrutiny.

In [6]:
from collections import Counter

assert len(vote_records) == N_PAIRS_EXPECTED * N_VOTES, len(vote_records)
key_counts = Counter(
    (record['Id'], record['frame_a'], record['frame_b'])
    for record in vote_records
)
assert len(key_counts) == N_PAIRS_EXPECTED, len(key_counts)
assert all(count == N_VOTES for count in key_counts.values())

for order in ('ab', 'ba'):
    records = [r for r in vote_records if r['presentation_order'] == order]
    n_fail = sum(not r['parse_ok'] for r in records)
    n_contra = sum(
        r['parse_ok'] and r['parsed_answer'] == r['frame_b'] for r in records
    )
    print(
        f'{order}: n={len(records)} parse_fail={n_fail / len(records):.4f} '
        f'contradiction_rate={n_contra / len(records):.4f}'
    )
print('validation ok:', len(key_counts), 'pairs x', N_VOTES, 'votes,',
      f'{main_wall_seconds / 60:.1f} min wall')


ab: n=4842 parse_fail=0.0006 contradiction_rate=0.1010
ba: n=4842 parse_fail=0.0000 contradiction_rate=0.7834
validation ok: 2421 pairs x 4 votes, 0.0 min wall


## Output

Copy these back into the repo:

- `snu ai/outputs/pairwise_requery/{votes.csv, metadata.json, pilot_votes.csv,
  pilot_metadata.json}` → `outputs/pairwise_requery/`
- optionally archive `votes_state.jsonl` alongside

Then run `python -m scripts.sweep_pairwise_requery` locally for the
tau × rule grid and `sweep_report.md`.